# Healthcare VOC Compliance Analysis

Analysis of VOC (Volatile Organic Compound) regulatory limits and cleaning product compliance across 26 North American jurisdictions. This notebook uses two datasets:

1. **VOC Regulatory Limits** — 650 records of jurisdiction-specific VOC limits for 25 cleaning product categories
2. **Healthcare Cleaning Products** — 5,000 products with VOC content, certifications, and per-jurisdiction compliance flags

Data sources: EPA 40 CFR Part 59, CARB Consumer Products Regulations, Canada SOR/2021-268, OSHA 29 CFR 1910.1000.

For a detailed walkthrough of the regulatory landscape and calculation methodology, see the full PDF guide: [VOC Compliance for Healthcare Facility Cleaning](https://www.binx.ca/guides/healthcare-voc-compliance-guide.pdf) (Binx Professional Cleaning).

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# Load datasets
limits_df = pd.read_csv('../datasets/voc_regulatory_limits.csv')
products_df = pd.read_csv('../datasets/healthcare_cleaning_products_voc.csv')

print(f'Regulatory limits: {len(limits_df)} records across {limits_df["jurisdiction"].nunique()} jurisdictions')
print(f'Products: {len(products_df)} records across {products_df["product_category"].nunique()} categories')
print(f'Healthcare-approved products: {(products_df["healthcare_approved"] == "Yes").sum()}')

## 1. VOC Limit Comparison Across Jurisdictions

How do VOC limits for the most common healthcare cleaning products compare across regulatory tiers?

In [ ]:
# Focus on key healthcare categories
key_categories = [
    'General Purpose Cleaner', 'Disinfectant (Spray)', 'Disinfectant (Concentrate)',
    'Glass Cleaner', 'Bathroom and Tile Cleaner', 'Floor Wax Stripper'
]

# Select representative jurisdictions
rep_jurisdictions = ['US-FED', 'US-CA', 'US-NY', 'US-MI', 'CA-FED', 'CA-ON']
rep_labels = ['EPA Federal', 'California (CARB)', 'New York (OTC)', 'Michigan', 'Canada Federal', 'Ontario']

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle('VOC Limits by Jurisdiction — Key Healthcare Cleaning Categories', fontsize=14, fontweight='bold')

for idx, cat in enumerate(key_categories):
    ax = axes[idx // 3][idx % 3]
    cat_data = limits_df[limits_df['product_category'] == cat]
    
    values = []
    for jcode in rep_jurisdictions:
        row = cat_data[cat_data['jurisdiction_code'] == jcode]
        values.append(row['voc_limit_g_per_L'].values[0] if len(row) > 0 else 0)
    
    colors = ['#1f77b4' if v > 10 else '#2ca02c' if v > 0 else '#7f7f7f' for v in values]
    bars = ax.barh(rep_labels, values, color=colors)
    ax.set_xlabel('VOC Limit (g/L)')
    ax.set_title(cat, fontsize=11)
    
    for bar, val in zip(bars, values):
        ax.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height()/2, 
                f'{val:.0f}', va='center', fontsize=9)

plt.tight_layout()
plt.savefig('jurisdiction_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 2. Product Compliance Heatmap

What percentage of products in each category are compliant in each jurisdiction tier?

In [ ]:
# Compute compliance rates by category and jurisdiction tier
tier_map = {
    'EPA Federal': 'US-FED', 'CARB (California)': 'US-CA',
    'OTC (New York)': 'US-NY', 'Canada Federal': 'CA-FED'
}

categories = products_df['product_category'].unique()
compliance_matrix = pd.DataFrame(index=categories, columns=tier_map.keys())

for cat in categories:
    cat_products = products_df[products_df['product_category'] == cat]
    for tier_name, jcode in tier_map.items():
        compliant_count = cat_products['compliant_jurisdictions'].str.contains(jcode, na=False).sum()
        compliance_matrix.loc[cat, tier_name] = round(compliant_count / len(cat_products) * 100, 1)

compliance_matrix = compliance_matrix.astype(float).sort_index()

fig, ax = plt.subplots(figsize=(12, 14))
im = ax.imshow(compliance_matrix.values, cmap='RdYlGn', aspect='auto', vmin=0, vmax=100)

ax.set_xticks(range(len(tier_map)))
ax.set_xticklabels(tier_map.keys(), rotation=45, ha='right')
ax.set_yticks(range(len(categories)))
ax.set_yticklabels(compliance_matrix.index, fontsize=9)

for i in range(len(compliance_matrix.index)):
    for j in range(len(tier_map)):
        ax.text(j, i, f'{compliance_matrix.iloc[i, j]:.0f}%', ha='center', va='center', fontsize=8)

ax.set_title('Product Compliance Rate by Category and Jurisdiction Tier (%)', fontsize=13, fontweight='bold')
plt.colorbar(im, ax=ax, label='Compliance Rate (%)', shrink=0.6)
plt.tight_layout()
plt.savefig('compliance_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. VOC Content Distribution by Product Category

In [ ]:
fig, ax = plt.subplots(figsize=(14, 8))

cat_order = products_df.groupby('product_category')['voc_content_g_per_L'].median().sort_values().index
products_df.boxplot(column='voc_content_g_per_L', by='product_category', ax=ax, vert=False,
                     positions=range(len(cat_order)))

ax.set_yticklabels(cat_order, fontsize=8)
ax.set_xlabel('VOC Content (g/L as concentrate)')
ax.set_title('VOC Content Distribution by Product Category', fontsize=13, fontweight='bold')
fig.suptitle('')

# Add CARB and EPA reference lines for General Purpose Cleaner
ax.axvline(x=4.0, color='green', linestyle='--', alpha=0.5, label='CARB limit (GP Cleaner: 4 g/L)')
ax.axvline(x=10.0, color='orange', linestyle='--', alpha=0.5, label='EPA limit (GP Cleaner: 10 g/L)')
ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig('voc_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Certification Coverage

In [ ]:
# Parse certifications
cert_names = ['Green Seal GS-37', 'UL ECOLOGO', 'EPA Safer Choice', 'UL GREENGUARD Gold', 'LEED v4 Compliant']

cert_counts = {}
for cert in cert_names:
    cert_counts[cert] = products_df['certifications'].str.contains(cert, na=False).sum()

cert_series = pd.Series(cert_counts).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(10, 5))
cert_series.plot(kind='barh', ax=ax, color='#2ca02c')
ax.set_xlabel('Number of Products')
ax.set_title('Products by Certification', fontsize=13, fontweight='bold')

for i, (val, name) in enumerate(zip(cert_series.values, cert_series.index)):
    pct = val / len(products_df) * 100
    ax.text(val + 20, i, f'{val} ({pct:.0f}%)', va='center', fontsize=9)

plt.tight_layout()
plt.savefig('certification_coverage.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. VOC Exposure Calculation — Sample Scenario

Calculate VOC exposure for a standard patient room cleaning cycle.

In [ ]:
import math

# Scenario: 200 sqft patient room, 9ft ceiling, 6 ACH
# Product: General Purpose Cleaner, 8 g/L VOC concentrate, 1:64 dilution

SQFT_TO_SQM = 0.09290304
FT_TO_M = 0.3048
OSHA_PEL = 300.0  # mg/m³

room_sqft = 200
ceiling_ft = 9
product_voc = 8.0  # g/L
dilution = 1/65  # 1:64
coverage = 400  # sqft/L
ach = 6.0

room_vol_m3 = room_sqft * SQFT_TO_SQM * ceiling_ft * FT_TO_M
effective_voc = product_voc * dilution
product_applied = room_sqft / coverage
total_voc_mg = effective_voc * product_applied * 1000

emission_rate = total_voc_mg / 0.5  # 30-min cleaning
vent_rate = ach * room_vol_m3
steady_state = emission_rate / vent_rate
osha_pct = steady_state / OSHA_PEL * 100

print('=== Patient Room VOC Exposure Calculation ===')
print(f'Room: {room_sqft} sqft × {ceiling_ft} ft = {room_vol_m3:.1f} m³')
print(f'Product: {product_voc} g/L concentrate, diluted 1:64 → {effective_voc:.4f} g/L')
print(f'Applied: {product_applied:.3f} L')
print(f'Total VOC released: {total_voc_mg:.1f} mg')
print(f'Steady-state concentration: {steady_state:.2f} mg/m³')
print(f'OSHA PEL usage: {osha_pct:.2f}%')
print(f'\nVerdict: {"SAFE" if osha_pct < 10 else "CAUTION" if osha_pct < 50 else "WARNING"}')

## 6. Summary Statistics

Full regulatory reference guide with jurisdiction comparison tables, ASHRAE ventilation rates, certification standards, and a Northern Ontario healthcare facility case study: [VOC Compliance for Healthcare Facility Cleaning (PDF)](https://www.binx.ca/guides/healthcare-voc-compliance-guide.pdf)

In [ ]:
print('=== Dataset Summary ===')
print(f'\nRegulatory Limits Dataset:')
print(f'  Jurisdictions: {limits_df["jurisdiction"].nunique()}')
print(f'  Product categories: {limits_df["product_category"].nunique()}')
print(f'  Total records: {len(limits_df)}')
print(f'  Strictest jurisdiction: {limits_df.groupby("jurisdiction")["voc_limit_g_per_L"].mean().idxmin()}')
print(f'  Most permissive: {limits_df.groupby("jurisdiction")["voc_limit_g_per_L"].mean().idxmax()}')

print(f'\nProduct Dataset:')
print(f'  Total products: {len(products_df)}')
print(f'  Healthcare-approved: {(products_df["healthcare_approved"] == "Yes").sum()} ({(products_df["healthcare_approved"] == "Yes").mean()*100:.0f}%)')
print(f'  IPAC compatible: {(products_df["ipac_compatible"] == "Yes").sum()}')
print(f'  With certifications: {(products_df["certifications"] != "None").sum()}')
print(f'  Mean VOC content: {products_df["voc_content_g_per_L"].mean():.1f} g/L')
print(f'  Median VOC content: {products_df["voc_content_g_per_L"].median():.1f} g/L')
print(f'  Products compliant in all 26 jurisdictions: {(products_df["compliant_jurisdiction_count"] == 26).sum()}')